In [1]:
from huggingface_hub import login
login(token="add your keyxxxx")

In [2]:
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import torch
from torch.utils.data import DataLoader
import os
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset,DatasetDict

2026-03-05 16:35:28.150588: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 16:35:29.117639: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-05 16:35:29.117689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-05 16:35:29.218224: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-05 16:35:29.308662: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructio

[2026-03-05 16:36:21,715] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: warning: libstdc++.so.6, needed by /home/mhossai5/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::string::substr(unsigned long, unsigned long) const@GLIBCXX_3.4'
/home/mhossai5/.conda/envs/llm_tor2/compiler_compat/ld: /home/mhossai5/local/cuda-11.8/lib64/libcufile.

In [4]:
# ds = load_dataset("zgcarvalho/oas-test",'unpaired')

In [5]:
# ds

In [6]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from transformers import PreTrainedTokenizerFast

# Define amino acid vocabulary including special tokens
amino_acids = ['[PAD]', '[BOS]', '[EOS]', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 
               'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y']

# Initialize BPE model with unknown token
tokenizer = Tokenizer(models.BPE(unk_token='[PAD]'))

# Set pre-tokenization (byte-level split)
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()

# Train the tokenizer with dummy data (repeat the vocabulary 100 times)
trainer = trainers.BpeTrainer(vocab=amino_acids, special_tokens=['[PAD]', '[BOS]', '[EOS]'])
tokenizer.train_from_iterator(["".join(amino_acids)], trainer=trainer)

# Wrap with Hugging Face PreTrainedTokenizerFast
protein_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
protein_tokenizer.add_special_tokens({
    'pad_token': '[PAD]',
    'bos_token': '[BOS]',
    'eos_token': '[EOS]'
})

0

Ignored unknown kwargs option vocab





In [5]:
def tokenize_function(example):
    return protein_tokenizer(
        ["[BOS]"+i+"[EOS]" for i in example["sequence"]],
        padding="max_length",
        truncation=True,
        max_length=174,
        return_tensors = 'pt',
        add_special_tokens=True  # Adds BOS and EOS
    )

In [6]:
# from datasets import load_dataset
# # Apply the tokenizer
# tokenized_ds = ds.map(tokenize_function, batched=True)

In [7]:
# tokenized_ds.save_to_disk("oas_tokenized_173")
from datasets import load_from_disk

tokenized_ds = load_from_disk("oas_tokenized_173")

Loading dataset from disk:   0%|          | 0/43 [00:00<?, ?it/s]

In [8]:
# Split the 'train' set into train/test
split_dataset = tokenized_ds["train"].train_test_split(test_size=0.1, seed=42)

# Wrap it back into a DatasetDict if you want
final_dataset = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})

In [14]:
import torch
from transformers import (
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling, 
    
)

In [9]:
#ref: https://huggingface.co/docs/transformers/en/model_doc/gemma3#transformers.Gemma3TextModel

from transformers import Gemma3TextConfig, Gemma3TextModel,Gemma3ForCausalLM

config = Gemma3TextConfig(
    vocab_size = protein_tokenizer.vocab_size,

    hidden_size = 384,
    intermediate_size = 1024,
    num_hidden_layers = 6,

    num_attention_heads = 4,
    num_key_value_heads = 4,
    head_dim = 96,   # required in Gemma

    hidden_activation = "gelu_pytorch_tanh",

    max_position_embeddings = 512,

    initializer_range = 0.02,
    rms_norm_eps = 1e-6,

    use_cache = True,
    use_bidirectional_attention = False,   # causal LM behavior

    pad_token_id = 0,
    bos_token_id = 1,
    eos_token_id = 2,

    attention_dropout = 0.0,
    attention_bias = False,

    sliding_window = 512,   # match your seq len

    tie_word_embeddings = False,
)

In [10]:
model = Gemma3ForCausalLM(config)#.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Assuming float32 (4 bytes per parameter)
model_size_gb = total_params * 4 / (1024 ** 3)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Estimated model size: {model_size_gb:.2f} GB")
model

Total parameters: 10,670,592
Trainable parameters: 10,670,592
Estimated model size: 0.04 GB


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(56, 384, padding_idx=0)
    (layers): ModuleList(
      (0-5): 6 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
          (q_norm): Gemma3RMSNorm((96,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((96,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=384, out_features=1024, bias=False)
          (up_proj): Linear(in_features=384, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=384, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((384,), eps=1e-06)
  

In [11]:
ckpt_path = "./checkpoints-gemma-3"
training_args = TrainingArguments(
    output_dir=ckpt_path,
    save_total_limit=3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_steps=500,
    logging_dir="./logs",
    logging_steps=10,
    bf16=True,
    report_to="wandb",
    
    save_strategy="epoch",
    eval_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",         # Track lowest validation loss
    greater_is_better=False                    # Because lower eval_loss is better
)

In [15]:
data_collator = DataCollatorForLanguageModeling(tokenizer=protein_tokenizer, mlm=False)
data_collator

DataCollatorForLanguageModeling(tokenizer=PreTrainedTokenizerFast(name_or_path='', vocab_size=56, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'pad_token': '[PAD]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), mlm=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt', seed=None)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset['train'],
    eval_dataset=final_dataset['test'],
    tokenizer=protein_tokenizer,
    data_collator=data_collator,
)
trainer.train()

/scratch/local/ipykernel_116958/1602760771.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: hossainstudy7 (hossainstudy7-freelancer). Use `wandb login --relogin` to force relogin


It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Epoch,Training Loss,Validation Loss
1,0.469900,0.472485
2,0.437300,0.444254


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

TrainOutput(global_step=83979, training_loss=0.47747853978513544, metrics={'train_runtime': 34191.2365, 'train_samples_per_second': 1257.583, 'train_steps_per_second': 2.456, 'total_flos': 4.780247940229202e+17, 'train_loss': 0.47747853978513544, 'epoch': 2.9999017637865597})

In [ ]:
trainer.save_model(ckpt_path+'-final')

In [19]:
from glob import glob

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# Path to your saved checkpoint folder
checkpoint_path = sorted(glob(ckpt_path))[-1]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Load model
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# Set model to evaluation mode and move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(58, 384, padding_idx=0)
    (layers): ModuleList(
      (0-5): 6 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=384, out_features=1024, bias=False)
          (up_proj): Linear(in_features=384, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=384, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((384,), eps=1e-06)
        (post_attention_layernorm): MistralRMSNorm((384,), eps=1e-06)
      )
    )
    (norm): MistralRMSNorm((384,), eps=1e-06)
    (

In [21]:
def generate_protein_sequence(
    prompt ='A',
    model  = model,
    tokenizer = protein_tokenizer,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
    top_k: int = 50,
    top_p: float = 0.95,
    repetition_penalty: float = 1.1,
    device: str = "cuda"  # or "cpu"
):
    model.eval()
    model.to(device)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate sequence
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=repetition_penalty,
        eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    )

    # Decode generated tokens (excluding the input prompt)
    generated_sequence = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(" ",'')
    
    return generated_sequence
